# **EDA**

In [1]:
import re
from pathlib import Path
from collections import Counter
import pandas as pd

In [2]:
df = pd.read_csv("data/raw/raw.csv", dtype={"id_artikel": str})
print(f"Artikel mentah: {len(df)}")
df.head(3)

Artikel mentah: 3979


,id_artikel,keyword,url,timestamp,judul,isi_mentah
0,777827,BCA,https://www.cnbcindonesia.com/market/202610201...,2026-10-20,"Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA ...",Indeks Harga Saham Gabungan (IHSG) berhasil me...
1,777666,BCA,https://www.cnbcindonesia.com/market/202610192...,2026-10-19,"RI Alami Halusinasi Daya Beli, Ekonom Ungkap F...",Kepala Ekonom PT Bank Central Asia Tbk David E...
2,777293,BCA,https://www.cnbcindonesia.com/market/202610190...,2026-10-19,"BCA Mau Buyback 19-21 Oktober, Karyawan Dilara...",PT Bank Central Asia Tbk (BBCA) menginformasik...


### **Gabungkan keyword yang relevan menjadi 1**

In [3]:
# BCA dan BBCA jadi satu artike

POLA_BANK = {
    "BCA":     r"\bbca\b|\bbbca\b|bank central asia", 
    "BRI":     r"\bbri\b|\bbbri\b|bank rakyat indonesia|brilink",
    "BNI":     r"\bbni\b|\bbbni\b|bank negara indonesia",
    "Mandiri": r"\bmandiri\b|\bbmri\b|bank mandiri",
}

def deteksi_bank(teks):
    teks = (teks or "").lower()
    return ", ".join(b for b, pat in POLA_BANK.items() if re.search(pat, teks))

df["bank"] = (df["judul"].fillna("") + " " + df["isi_mentah"].fillna("")).map(deteksi_bank)
print("Distribusi jumlah bank yang disebut per artikel:")
print(df["bank"].map(lambda x: len(x.split(", ")) if x else 0).value_counts().sort_index())
print("Artikel tanpa bank :", (df["bank"] == "").sum())
df[["id_artikel", "keyword", "bank", "judul"]].head(5)

Distribusi jumlah bank yang disebut per artikel:
bank
0       4
1    1868
2     754
3     908
4     445
Name: count, dtype: int64
Artikel tanpa bank : 4


,id_artikel,keyword,bank,judul
0,777827,BCA,"BCA, BRI, Mandiri","Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA ..."
1,777666,BCA,BCA,"RI Alami Halusinasi Daya Beli, Ekonom Ungkap F..."
2,777293,BCA,BCA,"BCA Mau Buyback 19-21 Oktober, Karyawan Dilara..."
3,776824,BCA,"BCA, BRI, BNI, Mandiri","Asing Keluar Rp2,92 Triliun di Sesi I, Saham B..."
4,776759,BCA,"BCA, BRI, BNI, Mandiri",Ada Bank Yang Sudah Jual Dolar di Rp18.300


### **Pecah berita menjadi per kalimat**

beberapa singkatan seperti PT. dispesifik, agar tidak terpisah kalimatnya

In [4]:
# Daftar singkatan yang titiknya JANGAN dianggap akhir kalimat.
# Ditambah gelar/sebutan yang muncul di data: Prof, Sdr, S.E, M.M, Ph.D.
SINGKATAN = ["PT", "Tbk", "Persero", "Rp", "US", "No", "Jl", "Dr", "Ir", "Hj",
             "Prof", "Sdr", "S.E", "M.M", "Ph.D",
             "dll", "dsb", "dkk", "a.l", "yakni"]

def pecah_kalimat(teks):
    t = re.sub(r"\s+", " ", str(teks)).strip()
    # lindungi titik pada singkatan (PT. Tbk. dst)
    for ab in SINGKATAN:
        t = re.sub(rf"\b{re.escape(ab)}\.", f"{ab}<DOT>", t)
    # lindungi inisial nama satu huruf kapital + titik (mis. "David E. Sumual", "M. Fauzan")
    t = re.sub(r"\b([A-Z])\.(?=\s)", r"\1<DOT>", t)
    # lindungi titik desimal & titik ribuan (12.5 ; 1.234)
    t = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", t)
    kalimat = re.split(r"(?<=[.!?])\s+", t)
    return [k.replace("<DOT>", ".").strip() for k in kalimat if k.strip()]

baris = []
for _, r in df.iterrows():
    for i, k in enumerate(pecah_kalimat(r["isi_mentah"])):
        baris.append({
            "id_artikel": r["id_artikel"],
            "id_kalimat": f"{r['id_artikel']}_{i}",
            "bank": r["bank"],
            "timestamp": r["timestamp"],
            "kalimat": k,
        })
kal = pd.DataFrame(baris)
kal["n_kata"] = kal["kalimat"].str.split().str.len()
print(f"Total kalimat mentah: {len(kal)} dari {len(df)} artikel")
print(f"Rata-rata kalimat/artikel: {len(kal)/len(df):.1f}")


Total kalimat mentah: 82368 dari 3979 artikel
Rata-rata kalimat/artikel: 20.7


### **Distribusi frekuensi kalimat**

In [5]:
print("Statistik jumlah kata per kalimat:")
print(kal["n_kata"].describe().round(1))

Statistik jumlah kata per kalimat:
count    82368.0
mean        19.2
std         14.0
min          1.0
25%         12.0
50%         17.0
75%         23.0
max        448.0
Name: n_kata, dtype: float64


In [6]:
print("Distribusi panjang kalimat (kata):")
bins = [0, 5, 10, 15, 20, 30, 50, 10000]
print(pd.cut(kal["n_kata"], bins=bins, right=False).value_counts().sort_index())

Distribusi panjang kalimat (kata):
n_kata
[0, 5)          1294
[5, 10)         8674
[10, 15)       21470
[15, 20)       20413
[20, 30)       21401
[30, 50)        7488
[50, 10000)     1628
Name: count, dtype: int64


### **Duplicate**

In [7]:
kal["kalimat"].duplicated().sum()

np.int64(17143)

In [8]:
kal[kal["kalimat"].duplicated() == True]

,id_artikel,id_kalimat,bank,timestamp,kalimat,n_kata
118,776155,776155_2,BCA,2026-10-14,"Sementara itu, untuk tenor menengah, seperti F...",19
119,776155,776155_3,BCA,2026-10-14,Sedangkan tenor panajang seperti FR0107 hanya ...,13
120,776155,776155_4,BCA,2026-10-14,"Dibanding tahun lalu, tepatnya hasil lelang pa...",27
121,776155,776155_5,BCA,2026-10-14,Sedangkan untuk tenor menengah seperti FR0109 ...,17
122,776155,776155_6,BCA,2026-10-14,Adapun untuk tenor panjang saat itu seperti FR...,23
...,...,...,...,...,...,...
82336,608753,608753_18,Mandiri,2025-02-07,Meskipun Kanada dan Meksiko mengalami penundaa...,10
82337,608753,608753_19,Mandiri,2025-02-07,Negeri Tirai Bambu kemudian membalas tarif imp...,8
82350,608753,608753_32,Mandiri,2025-02-07,CNBC INDONESIA RESEARCH,3
82357,608414,608414_6,Mandiri,2025-02-06,"Ditambah, dalam waktu dekat ini juga akan ada ...",19


In [9]:
print(kal.loc[kal["id_kalimat"] == "776155_2", "kalimat"].iloc[0])

Sementara itu, untuk tenor menengah, seperti FR0110 bid to cover ratio nya 1,94 kali dan FR0111 sebanyak 2,08 kali.


In [10]:
# Jumlah kalimat yang paling banyak terduplicate
dup_counts = kal["kalimat"].value_counts()
for s, c in dup_counts[dup_counts > 1].head(30).items():
    print(f"  ({c}x) {s[:80]}")

  (113x) 1.
  (99x) Selamat berinvestasi secara bijak.
  (76x) 2.
  (61x) 3.
  (54x) 4.
  (52x) CNBC INDONESIA RESEARCH market@cnbcindonesia.com
  (47x) Keputusan berinvestasi sepenuhnya berada di tangan masing-masing investor sesuai
  (44x) 5.
  (27x) Nasabah BRI Prioritas bisa mendapatkan BRI Prioritas Debit Card yang menyediakan
  (26x) 6.
  (23x) Mengutip Stockbit, berikut net foreign sell perdagangan Rabu!
  (18x) "Perusahaan para cukong dan keluarga Soeharto merupakan sasaran utama pembakaran
  (18x) "Dalam sekejap, seluruh mobil di garasi terbakar, termasuk juga seisi rumah.
  (18x) Nasabah istimewa BNI Emerald akan mendapatkan produk dan layanan yang dirancang 
  (18x) Keuntungan bagi nasabah prioritas Bank Mandiri dapat dirasakan sejak bergabung.
  (18x) Yakni welcome bonus berupa cashback senilai Rp 1,5 juta setelah aktivasi dan tra
  (18x) Kemudian, manfaat spesial bagi nasabah Mandiri Prioritas adalah 4x Livin'poin un
  (18x) Selain itu, ada perlindungan dari pembelian bara

beberapa kaliamt yang terduplicate merupakan kutipan dari seseorang / istilah umum, sehingga perlu didrop

### **Missing Text**

In [11]:
(kal["kalimat"].str.strip() == "").sum()

np.int64(0)

### **Kalimat dibawah 5**

In [12]:
n_pendek = (kal["n_kata"] < 5).sum()
print(f"Kalimat < 5 kata : {n_pendek} ({n_pendek/len(kal)*100:.1f}%)")

Kalimat < 5 kata : 1294 (1.6%)


In [13]:
pendek = kal[kal["n_kata"] <= 5]
print(f"Distribusi jumlah kata pada kalimat pendek: {dict(pendek['n_kata'].value_counts().sort_index())}")

Distribusi jumlah kata pada kalimat pendek: {1: np.int64(497), 2: np.int64(85), 3: np.int64(208), 4: np.int64(504), 5: np.int64(515)}


In [14]:
print("\n30 contoh kalimat < 5 kata (pastikan memang sampah, bukan relevan):")
for s in pendek["kalimat"].head(50).tolist():
    print("  |", s)


30 contoh kalimat < 5 kata (pastikan memang sampah, bukan relevan):
  | "Karena apa?
  | 2.
  | BCA menjadi yang terparah.
  | Prediksi itu kemudian benar terjadi.
  | Anthony tak berkutik.
  | (Kompas, 15 Mei 1998).
  | Pengalaman serupa dibagikan akun @mawarniseptiana.
  | laporannya ke official IG Dana.
  | Hati-hati akun palsu ya!" tulisnya.
  | 1.
  | 2.
  | 2.
  | 3.
  | Salah satunya, Mall Taman Anggrek.
  | 4.
  | 5.
  | 6.
  | 7.
  | 8.
  | 2.
  | Berikut kisah perjalanan keduanya.
  | "Yang pasti rekam jejak beliau.
  | Warisan Jumbo Michael Hartono!
  | Belakangan, keyakinan itu seolah terbukti.
  | Belakangan, keyakinan itu seolah terbukti.
  | "Itu kan sovereign rating.
  | "Enggak.
  | Alhasil Rupiah melemah.
  | 20% capital goods.
  | Sisanya baru consumer goods.
  | BCA menjadi yang terparah.
  | Prediksi itu kemudian benar terjadi.
  | Anthony tak berkutik.
  | (Kompas, 15 Mei 1998).
  | Namun, pembacaan itu perlu hati-hati.
  | Kedua, kelompok masyarakat bawah.
  | "

Ada beberapa kata dibawah 5, yang memang bermakna ("BCA menjadi yang terparah."). dan untuk kata - kata lain yang hanya 5 kata saja bermakna, sehingga filternya hanya akan menghapus data < 4

### **Pembersihan kalimat (sebelum BERTopic)**\n\nBerdasarkan EDA di atas: buang boilerplate/byline, kalimat sangat panjang (listing/tabel), kalimat terlalu pendek, lalu buang duplikat kalimat. Kalimat kutipan ('Mengutip/Melansir') **tidak** dibuang karena isinya informatif.

In [18]:
AWAL = len(kal)

# 0) Normalisasi karakter tak-terlihat & em-dash pembuka (sisa dateline)
#    - buang zero-width / LTR mark (\u200e, \u2060, \ufeff, \u200b)
#    - buang em-dash / hyphen yang menggantung di awal kalimat (sisa potongan dateline)
kal["kalimat"] = (kal["kalimat"]
    .str.replace("[\u200e\u200b\u2060\ufeff]", "", regex=True)       # <- HAPUS huruf r di depan
    .str.replace("^\\s*[\u2014\u2013\\-\u2022\u00b7]+\\s*", "", regex=True)
    .str.replace("\\s+", " ", regex=True)
    .str.strip())

# 1) Buang boilerplate / byline / navigasi
POLA_SAMPAH = re.compile(
    r"CNBC\s*INDONESIA\s*RESEARCH"
    r"|market@cnbcindonesia\.com"
    r"|Selamat\s+berinvestasi\s+secara\s+bijak"
    r"|Baca\s+halaman\s+(?:berikut|selanjut)"
    r"|\bNEXT\b"
    r"|Lihat\s*juga"
    r"|Saksikan\s+(?:Video|juga)"
    r"|Disclaimer|Sanggahan"
    r"|\bBaca\s*:"                       # sisa link "Baca:" yang menempel judul artikel lain
    r"|^Foto\s*:|^Infografis\s*:"        # caption gambar
    r"|^\(?Dok\b",                       # "(Dok BEI) Foto: ..."
    re.I,
)
mask_sampah = kal["kalimat"].str.contains(POLA_SAMPAH)
# penomoran list murni: "1.", "2)", "3 ."
mask_nomor = kal["kalimat"].str.match(r"^\s*\d+\s*[.)]?\s*$")
# baris rekomendasi/tabel saham: banyak "Buy/TP/SL" & pemisah "|" (listing, bukan kalimat)
mask_rekom = (kal["kalimat"].str.contains(r"\bRekomendasi Saham\b", case=False)
              | (kal["kalimat"].str.count(r"\|") >= 2)
              | (kal["kalimat"].str.contains(r"\bTP\b.*\bSL\b")))
# baris tabel suku bunga: diawali operator pembanding + banyak persen
mask_tabel = kal["kalimat"].str.match("^\\s*[<>\u2264\u2265]") & (kal["kalimat"].str.count("%") >= 2)

buang1 = int((mask_sampah | mask_nomor | mask_rekom | mask_tabel).sum())
kal = kal[~(mask_sampah | mask_nomor | mask_rekom | mask_tabel)].copy()
kal["n_kata"] = kal["kalimat"].str.split().str.len()   # hitung ulang (teks berubah di langkah 0)
print(f"1. Boilerplate/byline/penomoran/rekomendasi/tabel dibuang : {buang1}")

# 2) Buang kalimat sangat panjang (>80 kata) -> biasanya listing/tabel yang nempel jadi 1 kalimat
buang2 = int((kal["n_kata"] > 80).sum())
kal = kal[kal["n_kata"] <= 80].copy()
print(f"2. Kalimat > 80 kata (listing/tabel)                      : {buang2}")

# 3) Buang kalimat terlalu pendek (<4 kata) -> fragmen tak bermakna
buang3 = int((kal["n_kata"] < 4).sum())
kal = kal[kal["n_kata"] >= 4].copy()
print(f"3. Kalimat < 4 kata (fragmen)                             : {buang3}")

# 4) Buang duplikat kalimat identik (sisakan 1); paling berdampak untuk BERTopic
sebelum_dedup = len(kal)
kal = kal.drop_duplicates(subset="kalimat").copy()
buang4 = sebelum_dedup - len(kal)
print(f"4. Duplikat kalimat identik dibuang                       : {buang4}")

print(f"\nKalimat: {AWAL} -> {len(kal)} (total dibuang {AWAL-len(kal)})")


1. Boilerplate/byline/penomoran/rekomendasi/tabel dibuang : 920
2. Kalimat > 80 kata (listing/tabel)                      : 842
3. Kalimat < 4 kata (fragmen)                             : 343
4. Duplikat kalimat identik dibuang                       : 16448

Kalimat: 82368 -> 63815 (total dibuang 18553)


### **Frekuensi kata**

In [19]:
tokens = []
for k in kal["kalimat"]:
    tokens += re.findall(r"[a-zA-Z]+", str(k).lower())
freq = Counter(tokens)
print(f"Total token: {len(tokens)} | kata unik: {len(freq)}")
print("\n30 kata paling sering:")
for w, c in freq.most_common(30):
    print(f"  {w:18} {c}")

Total token: 1132207 | kata unik: 20713

30 kata paling sering:
  dan                23389
  yang               22952
  di                 17913
  rp                 17784
  saham              17634
  dengan             13191
  pada               12990
  ini                12300
  bank               12129
  dari               10688
  miliar             9468
  pt                 9441
  indonesia          9213
  dalam              8830
  tbk                8145
  menjadi            8139
  triliun            7171
  untuk              6792
  ihsg               6787
  pasar              6656
  indeks             6625
  ke                 6360
  itu                6111
  sebesar            6023
  juga               5955
  perdagangan        5575
  akan               5230
  tersebut           5085
  atau               4894
  hari               4691


### **Analisis stopword & casefolding**

In [20]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
stopwords = set(StopWordRemoverFactory().get_stop_words())

n_stop = sum(1 for t in tokens if t in stopwords)

contoh = kal["kalimat"].iloc[2]
print("asli        :", contoh[:100])
print("casefold    :", contoh.lower()[:100])

tanpa_stop = " ".join(w for w in re.findall(r"\w+", contoh.lower()) if w not in stopwords)
print("tanpa stop  :", tanpa_stop[:100])

asli        : Namun, asing masih saja mencatatkan penjualan dengan nilai yang cukup besar.
casefold    : namun, asing masih saja mencatatkan penjualan dengan nilai yang cukup besar.
tanpa stop  : asing mencatatkan penjualan nilai cukup besar


### **Simpan hasil ke `sentences.csv`**

In [21]:
out = Path("data/processed/sentences.csv")
out.parent.mkdir(parents=True, exist_ok=True)
kal[["id_artikel", "id_kalimat", "bank", "timestamp", "kalimat", "n_kata"]].to_csv(out, index=False)
print(f"Tersimpan {len(kal)} kalimat bersih -> {out}")


Tersimpan 63815 kalimat bersih -> data\processed\sentences.csv


## 7. Kesimpulan EDA

* **Sumber**: `raw.csv` = **3.979 artikel**, 100% kanal **market**, tanpa id duplikat / field kosong, rentang 21 Okt 2024 - 20 Okt 2026.
* **Pemecahan kalimat**: dilindungi untuk singkatan (PT, Tbk, Rp, gelar) **dan inisial nama satu huruf** (mis. "David E.", "M. Fauzan") agar tidak salah potong, serta titik desimal/ribuan.
* **Pembersihan sebelum BERTopic** (angka final tercetak saat run): dari **82.368** kalimat mentah menjadi **63.815** kalimat bersih. Tahap:
    * normalisasi karakter tak-terlihat (zero-width) & em-dash pembuka sisa dateline;
    * buang boilerplate/byline, penomoran list, **baris rekomendasi/tabel saham** (pola "Buy/TP/SL", banyak "|"), **caption gambar** ("Foto:/Infografis:/Dok"), sisa link **"Baca:"**, dan baris tabel suku bunga;
    * buang kalimat **> 80 kata** (listing/tabel) dan **< 4 kata** (fragmen);
    * buang **duplikat kalimat identik** (~16,4 ribu) - langkah paling berdampak untuk topic modeling.
* **Kalimat kutipan ("Mengutip/Melansir", ~1.291)**: **TIDAK dibuang** - isinya kalimat penuh bermuatan data/sentimen, hanya frasa atribusi di depan.
* **Tidak** dilakukan stemming / stopword-removal / lowercasing pada teks yang akan di-embed BERTopic & diklasifikasi, agar urutan kata dan negasi tetap utuh.
* **Output**: `data/processed/sentences.csv` (kolom: id_artikel, id_kalimat, bank, timestamp, kalimat, n_kata) - input untuk Fase 03 (BERTopic).
